<a href="https://colab.research.google.com/github/huseyincenik/john_snow_labs/blob/main/generating_conll_files_from_pretrained_models/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Training - Custom NER Model Training

This notebook trains a custom NER model using CoNLL format data.

**Google Drive Integration:**
- All files are saved to Google Drive
- Files are read from Google Drive
- Trained models are saved to Google Drive
- All code is embedded in this notebook (no external Python files required)

## Steps:
1. **Google Drive Connection** - Mount Google Drive
2. **Setup & License** - Spark NLP Healthcare license and environment setup
3. **CoNLL Dataset Loading** - Load CoNLL training data from Google Drive
4. **Train/Validation Split** - Split data into training and validation sets
5. **Model Training** - Train custom NER model
6. **Model Evaluation** - Evaluate model with precision, recall, F1-score metrics
7. **Model Saving** - Save trained model to Google Drive

**Requirements:**
- `data_prep.ipynb` notebook must be run first
- CoNLL file must exist at `data/conll/conll2003_text_file.conll` in Google Drive


## 1. Google Drive Connection


In [1]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path


# Mount Google Drive
drive.mount('/content/drive')

# Set project folder in Google Drive
PROJECT_FOLDER = '/content/drive/MyDrive/john_snow_labs_ner'
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Change working directory
os.chdir(PROJECT_FOLDER)

# Create folder structure
for folder in ['models/trained', 'ner_logs', 'data/processed']:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"✅ Project folder: {PROJECT_FOLDER}")
print(f"✅ Current directory: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Project folder: /content/drive/MyDrive/john_snow_labs_ner
✅ Current directory: /content/drive/MyDrive/john_snow_labs_ner


## 2. Setup & License Configuration


In [2]:
import json
import os

# Load license keys from Google Drive
license_path = f'{PROJECT_FOLDER}/spark_jsl.json'
if not os.path.exists(license_path):
    print("❌ License file not found!")
    print("Please run data_prep.ipynb first or upload spark_jsl.json to Google Drive")
    raise FileNotFoundError(f"License file not found at {license_path}")

with open(license_path) as f:
    license_keys = json.load(f)

# Set license keys as environment variables
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.2.1
Public Version: 6.2.0


In [3]:
# Install Java (required for Spark)

import subprocess
import os

try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java is already installed: {java_version.split(chr(10))[0]}")

    if 'JAVA_HOME' not in os.environ:
        java_paths = [
            "/usr/lib/jvm/java-11-openjdk-amd64",
            "/usr/lib/jvm/java-8-openjdk-amd64",
            "/usr/lib/jvm/default-java"
        ]
        for path in java_paths:
            if os.path.exists(path):
                os.environ["JAVA_HOME"] = path
                print(f"✅ Set JAVA_HOME to: {path}")
                break
except Exception as e:
    print(f"Java check failed: {e}")
    print("Installing Java 11...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✅ Java 11 installation attempted")


# Check GPU availability safely
try:
    gpu_check = subprocess.run(
        ['nvidia-smi'],
        capture_output=True,
        text=True
    )
    has_gpu = gpu_check.returncode == 0
except FileNotFoundError:
    has_gpu = False

if has_gpu:
    print("🚀 GPU detected! Installing PyTorch with CUDA support...")
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
else:
    print("⚠️ GPU not detected. Installing PyTorch CPU version...")
    %pip install -q torch torchvision torchaudio


# Install PySpark and Spark NLP
%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Install Spark NLP Healthcare
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Install additional dependencies
%pip install -q pandas numpy tqdm requests

print("✅ All libraries installed successfully!")
if has_gpu:
    print("✅ GPU-accelerated PyTorch installed")


✅ Java is already installed: openjdk version "17.0.16" 2025-07-15
⚠️ GPU not detected. Installing PyTorch CPU version...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spark-nlp-jsl 6.2.1 requires spark-nlp==6.2.2, but you have spark-nlp 6.2.0 which is incompatible.
✅ All libraries installed successfully!


In [4]:
import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import WordEmbeddingsModel
from sparknlp_jsl.annotator import (
    MedicalNerApproach,
    MedicalNerDLGraphChecker,
    MedicalNerModel
)
from sparknlp.training import CoNLL
from sparknlp_jsl.eval import NerDLMetrics
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🚀 GPU Detected: {gpu_name}")
    else:
        print("⚠️  No GPU detected. Using CPU mode.")
except ImportError:
    print("⚠️  PyTorch not available. GPU check skipped.")
    gpu_available = False

# Spark configuration
params = {
    "spark.driver.memory": "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled in Spark configuration")

# Start Spark session
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")

    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized successfully")

except Exception as e:
    print(f"❌ Error starting Spark session: {e}")
    raise

spark


⚠️  No GPU detected. Using CPU mode.
Starting Spark session...
✅ Spark NLP Version: 6.2.2
✅ Spark NLP JSL Version: 6.2.1
✅ Spark session initialized successfully


## 3. Model Training Functions (Embedded)


In [5]:
# Model Training Functions - All code embedded in notebook

def load_embeddings(embeddings_model="embeddings_clinical"):
    """Load clinical word embeddings"""
    print(f"Loading {embeddings_model} embeddings...")
    clinical_embeddings = WordEmbeddingsModel.pretrained(
        embeddings_model, "en", "clinical/models"
    ).setInputCols(["sentence", "token"]).setOutputCol("embeddings")
    print("✅ Embeddings loaded successfully!")
    return clinical_embeddings

def load_conll_dataset(conll_path):
    """Load CoNLL format dataset"""
    print(f"Loading CoNLL dataset from {conll_path}...")
    data = CoNLL().readDataset(spark, conll_path)
    print(f"✅ Dataset loaded: {data.count()} sentences")
    return data

def split_dataset(data, train_ratio=0.8, seed=100):
    """Split dataset into train and validation sets"""
    train_data, validation_data = data.randomSplit(
        [train_ratio, 1 - train_ratio], seed=seed
    )
    print(f"✅ Train set: {train_data.count()} sentences")
    print(f"✅ Validation set: {validation_data.count()} sentences")
    return train_data, validation_data

def create_training_pipeline(clinical_embeddings,
                            max_epochs=20,
                            lr=0.003,
                            batch_size=8,
                            random_seed=0,
                            verbose=1,
                            test_dataset=None,
                            output_logs_path="./ner_logs",
                            validation_split=0.1,
                            use_best_model=True,
                            early_stopping_criterion=0.04,
                            early_stopping_patience=3):
    """Create training pipeline"""
    # Graph checker
    ner_dl_graph_checker = MedicalNerDLGraphChecker()\
        .setInputCols(["sentence", "token"])\
        .setLabelColumn("label")\
        .setEmbeddingsModel(clinical_embeddings)

    # NER Tagger
    ner_tagger = MedicalNerApproach()\
        .setInputCols(["sentence", "token", "embeddings"])\
        .setLabelColumn("label")\
        .setOutputCol("ner")\
        .setMaxEpochs(max_epochs)\
        .setLr(lr)\
        .setBatchSize(batch_size)\
        .setRandomSeed(random_seed)\
        .setVerbose(verbose)\
        .setEvaluationLogExtended(True)\
        .setEnableOutputLogs(True)\
        .setIncludeConfidence(True)\
        .setValidationSplit(validation_split)\
        .setUseBestModel(use_best_model)\
        .setEarlyStoppingCriterion(early_stopping_criterion)\
        .setEarlyStoppingPatience(early_stopping_patience)\
        .setOutputLogsPath(output_logs_path)

    if test_dataset:
        ner_tagger.setTestDataset(test_dataset)

    pipeline = Pipeline(stages=[
        clinical_embeddings,
        ner_dl_graph_checker,
        ner_tagger
    ])

    return pipeline

def evaluate_model(trained_model, test_data, clinical_embeddings, drop_o=True, case_sensitive=True):
    """Evaluate model performance"""
    print("Evaluating model...")
    pred_df = trained_model.stages[2].transform(
        clinical_embeddings.transform(test_data)
    )

    evaler = NerDLMetrics(mode="full_chunk")
    eval_result = evaler.computeMetricsFromDF(
        pred_df.select("label", "ner"),
        prediction_col="ner",
        label_col="label",
        drop_o=drop_o,
        case_sensitive=case_sensitive
    ).cache()

    # Format results
    eval_result_formatted = eval_result.withColumn(
        "precision", F.round(eval_result["precision"], 4)
    ).withColumn(
        "recall", F.round(eval_result["recall"], 4)
    ).withColumn(
        "f1", F.round(eval_result["f1"], 4)
    )

    print("\nEvaluation Results:")
    eval_result_formatted.show(100)

    # Calculate macro and micro averages
    print("\nMacro Average F1:")
    eval_result.selectExpr("avg(f1) as macro").show()
    print("\nMicro Average F1:")
    eval_result.selectExpr(
        "sum(f1*total) as sumprod", "sum(total) as sumtotal"
    ).selectExpr("sumprod/sumtotal as micro").show()

    return eval_result_formatted

def save_model(trained_model, model_path):
    """Save trained model"""
    model_path = Path(model_path)
    model_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Saving model to {model_path}...")
    trained_model.stages[2].write().overwrite().save(str(model_path))
    print("✅ Model saved successfully!")

print("✅ Model training functions defined")


✅ Model training functions defined


## 4. Load CoNLL Dataset and Embeddings


In [6]:
# Load clinical embeddings
clinical_embeddings = load_embeddings("embeddings_clinical")

# Load CoNLL dataset from Google Drive
conll_path = f"{PROJECT_FOLDER}/data/conll/conll2003_text_file.conll"
if not os.path.exists(conll_path):
    print(f"❌ CoNLL file not found at {conll_path}")
    print("Please run data_prep.ipynb first to generate the CoNLL file")
    raise FileNotFoundError(f"CoNLL file not found at {conll_path}")

training_data = load_conll_dataset(conll_path)


Loading embeddings_clinical embeddings...
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
✅ Embeddings loaded successfully!
Loading CoNLL dataset from /content/drive/MyDrive/john_snow_labs_ner/data/conll/conll2003_text_file.conll...
✅ Dataset loaded: 25966 sentences


## 5. Split Dataset into Train and Validation


In [7]:
# Split dataset into train and validation
train_data, validation_data = split_dataset(training_data, train_ratio=0.8, seed=100)

# Save validation data as parquet for evaluation
test_data_path = f"{PROJECT_FOLDER}/data/processed/test_data.parquet"
Path(test_data_path).parent.mkdir(parents=True, exist_ok=True)
clinical_embeddings.transform(validation_data).write.mode("overwrite").parquet(test_data_path)
print(f"✅ Validation data saved to {test_data_path}")


✅ Train set: 20768 sentences
✅ Validation set: 5198 sentences
✅ Validation data saved to /content/drive/MyDrive/john_snow_labs_ner/data/processed/test_data.parquet


## 6. Model Training


In [8]:
# GPU optimization for batch size
try:
    import torch
    use_gpu = torch.cuda.is_available()
    if use_gpu:
        batch_size = 16
        print("🚀 GPU detected - using optimized batch size")
    else:
        batch_size = 8
        print("Using CPU mode - standard batch size")
except:
    batch_size = 8
    use_gpu = False

# Create training pipeline
output_logs_path = f"{PROJECT_FOLDER}/ner_logs"
training_pipeline = create_training_pipeline(
    clinical_embeddings=clinical_embeddings,
    max_epochs=35,
    lr=0.001,
    batch_size=batch_size,
    random_seed=0,
    verbose=1,
    test_dataset=test_data_path,
    output_logs_path=output_logs_path,
    validation_split=0.2,
    use_best_model=True,
    early_stopping_criterion=0.02,
    early_stopping_patience=6
)

print("✅ Training pipeline created")
print(f"Training parameters:")
print(f"  - Max epochs: 20")
print(f"  - Learning rate: 0.001")
print(f"  - Batch size: {batch_size} {'(GPU optimized)' if use_gpu else '(CPU)'}")
print(f"  - Validation split: 0.2")
print(f"  - Early stopping: Enabled")


Using CPU mode - standard batch size
✅ Training pipeline created
Training parameters:
  - Max epochs: 20
  - Learning rate: 0.001
  - Batch size: 8 (CPU)
  - Validation split: 0.2
  - Early stopping: Enabled


In [9]:
spark.range(1).show()


+---+
| id|
+---+
|  0|
+---+



In [10]:
# Start model training
try:
    import torch
    if torch.cuda.is_available():
        print("🚀 Starting model training with GPU acceleration...")
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Starting model training on CPU... This may take several minutes...")
except:
    print("Starting model training... This may take several minutes...")

trained_model = training_pipeline.fit(train_data)

print("\n✅ Model training completed!")
print(f"\nTraining logs saved to {output_logs_path}/")


Starting model training on CPU... This may take several minutes...

✅ Model training completed!

Training logs saved to /content/drive/MyDrive/john_snow_labs_ner/ner_logs/


## 6.1 View Training Logs

View the training logs to see the training progress and metrics.

In [11]:
# View training logs
import glob

log_files = glob.glob(f"{PROJECT_FOLDER}/ner_logs/MedicalNerApproach*")
if log_files:
    latest_log = max(log_files, key=os.path.getctime)
    print(f"Latest training log: {latest_log}\n")
    with open(latest_log, 'r') as f:
        print(f.read())
else:
    print("No training logs found.")

Latest training log: /content/drive/MyDrive/john_snow_labs_ner/ner_logs/MedicalNerApproach_336e4d92d530.log

Name of the selected graph: medical-ner-dl/blstm_25_200_128_128.pb
Training started - total epochs: 35 - lr: 0.001 - batch size: 8 - labels: 23 - chars: 85 - training examples: 16614


Epoch 1/35 started, lr: 0.001, dataset size: 16614


Epoch 1/35 - 163.32s - loss: 10416.953 - avg training loss: 5.022639 - batches: 2074
Quality on validation dataset (20.0%), validation examples = 4154
time to finish evaluation: 12.88s
Total validation loss: 1132.3306	Avg validation loss: 2.1527
label	 tp	 fp	 fn	 prec	 rec	 f1
B-DRUG	 226	 37	 80	 0.8593156	 0.7385621	 0.7943761
I-NAME	 0	 0	 19	 0.0	 0.0	 0.0
I-AGE	 0	 0	 1	 0.0	 0.0	 0.0
I-TREATMENT	 2340	 412	 493	 0.8502907	 0.82597953	 0.8379588
B-DATE	 131	 24	 14	 0.8451613	 0.9034483	 0.87333333
I-DATE	 25	 10	 4	 0.71428573	 0.86206895	 0.78124994
I-LOCATION	 25	 0	 25	 1.0	 0.5	 0.6666667
I-PROBLEM	 4109	 342	 1242	 0.92316335	 0.7678

## 6.2 Evaluate Model Performance

Evaluate the trained model on the test set using precision, recall, and F1-score metrics.

In [12]:
# Evaluate model performance
eval_results = evaluate_model(trained_model, validation_data, clinical_embeddings,
                              drop_o=True, case_sensitive=True)

print("\n✅ Evaluation completed!")


Evaluating model...

Evaluation Results:
+----------+------+-----+-----+------+---------+------+------+
|    entity|    tp|   fp|   fn| total|precision|recall|    f1|
+----------+------+-----+-----+------+---------+------+------+
|      NAME|  48.0|  1.0|  9.0|  57.0|   0.9796|0.8421|0.9057|
|   PROBLEM|3203.0|445.0|475.0|3678.0|    0.878|0.8709|0.8744|
|      DATE| 156.0|  8.0| 13.0| 169.0|   0.9512|0.9231|0.9369|
|        ID|   4.0|  2.0|  1.0|   5.0|   0.6667|   0.8|0.7273|
|      DRUG| 335.0| 22.0| 61.0| 396.0|   0.9384| 0.846|0.8898|
|    DOSAGE|  29.0| 10.0| 12.0|  41.0|   0.7436|0.7073| 0.725|
|  LOCATION|  27.0|  6.0| 11.0|  38.0|   0.8182|0.7105|0.7606|
| TREATMENT|1889.0|335.0|427.0|2316.0|   0.8494|0.8156|0.8322|
|      TEST|1001.0|196.0|145.0|1146.0|   0.8363|0.8735|0.8545|
|PROFESSION|   5.0|  1.0|  7.0|  12.0|   0.8333|0.4167|0.5556|
|       AGE|  76.0|  3.0| 13.0|  89.0|    0.962|0.8539|0.9048|
+----------+------+-----+-----+------+---------+------+------+


Macro Averag

## 6.3 Save Trained Model

Save the trained model for future use.

In [13]:
# Save model
model_path = f"{PROJECT_FOLDER}/models/trained/custom_ner_model"
save_model(trained_model, model_path)

print(f"\n✅ Model saved to {model_path}")

Saving model to /content/drive/MyDrive/john_snow_labs_ner/models/trained/custom_ner_model...
✅ Model saved successfully!

✅ Model saved to /content/drive/MyDrive/john_snow_labs_ner/models/trained/custom_ner_model


## Summary

✅ **Model training completed!**

**Training Results:**
- Dataset:  **25,966 sentences** (80% train / 20% validation)
- Epochs: **13/35** (early stopping triggered after 13 epochs)
- **Macro F1: 81.51%** | **Micro F1: 86.00%**

**Best Performing Entities:**
- **DATE**: 93.69% F1
- **AGE**: 90.48% F1
- **NAME**: 90.57% F1
- **PROBLEM**: 87.44% F1
- **DRUG**:  88.98% F1
- **TEST**: 85.45% F1

**Model Performance (Final Epoch - Validation Set):**
| Entity | Precision | Recall | F1-Score |
|--------|-----------|--------|----------|
| DATE | 95.12% | 92.31% | 93.69% |
| AGE | 96.20% | 85.39% | 90.48% |
| NAME | 94.12% | 84.21% | 88.89% |
| PROBLEM | 92.54% | 90.70% | 91.61% |
| DRUG | 92.47% | 86.87% | 89.58% |
| TREATMENT | 87.08% | 87.00% | 87.04% |
| TEST | 89.55% | 81.01% | 85.06% |
| LOCATION | 86.49% | 84.21% | 85.33% |

**Saved Files:**
- `models/trained/custom_ner_model` - Trained NER model
- `data/processed/test_data.parquet` - Validation dataset
- `ner_logs/MedicalNerApproach_*. log` - Training logs

**Next step:** Load and use the trained model for NER predictions.